# Modelagem — risco de defasagem e risco de evasão

Datathon Fase 5 — Passos Mágicos. Dois modelos (XGBoost) a partir do painel
longitudinal PEDE 2022-2024 (`docs/decisions/0001-consolidacao-painel.md`):

- **`piora_defasagem`**: defasagem piora do ano `t` para `t+1`
- **`evasao`**: aluno ausente em `t+1` (excluída saída natural por fase/idade)

Split temporal: treino nas transições 2022→2023, teste em 2023→2024 — sem
vazamento, reflete o uso real (prever o próximo ano com dados do ano corrente).

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score, classification_report, roc_auc_score,
)

pd.set_option("display.width", 120)
np.random.seed(42)

In [2]:
feats = pd.read_csv("../data/processed/features.csv")

FEATURE_COLS = [c for c in feats.columns if c not in ("ra", "ano", "piora_defasagem", "evasao")]
CAT_COLS = ["genero"]

for c in CAT_COLS:
    feats[c] = feats[c].astype("category")

train = feats[feats["ano"] == 2022]
test = feats[feats["ano"] == 2023]
print(f"treino: {len(train)} linhas | teste: {len(test)} linhas")
print(f"features: {len(FEATURE_COLS)}")

treino: 860 linhas | teste: 1014 linhas
features: 25


In [3]:
def train_eval(target: str, train: pd.DataFrame, test: pd.DataFrame):
    """Treina XGBoost (missing nativo, sem imputação) e avalia no split temporal."""
    tr = train.dropna(subset=[target])
    te = test.dropna(subset=[target])
    X_tr, y_tr = tr[FEATURE_COLS], tr[target].astype(int)
    X_te, y_te = te[FEATURE_COLS], te[target].astype(int)

    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum(),
        enable_categorical=True, eval_metric="aucpr", random_state=42,
    )
    model.fit(X_tr, y_tr)

    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"=== {target} ===")
    print(f"treino: {len(X_tr)} ({y_tr.mean():.1%} positivo) | teste: {len(X_te)} ({y_te.mean():.1%} positivo)")
    print(f"ROC-AUC: {roc_auc_score(y_te, proba):.3f} | PR-AUC: {average_precision_score(y_te, proba):.3f}")
    print(classification_report(y_te, pred, digits=2))
    return model, X_tr, X_te, te

In [4]:
model_defasagem, Xtr_def, Xte_def, te_def = train_eval("piora_defasagem", train, test)

=== piora_defasagem ===
treino: 600 (17.3% positivo) | teste: 765 (17.3% positivo)
ROC-AUC: 0.834 | PR-AUC: 0.516
              precision    recall  f1-score   support

           0       0.88      0.93      0.90       633
           1       0.54      0.39      0.46       132

    accuracy                           0.84       765
   macro avg       0.71      0.66      0.68       765
weighted avg       0.82      0.84      0.83       765



In [5]:
model_evasao, Xtr_ev, Xte_ev, te_ev = train_eval("evasao", train, test)

=== evasao ===
treino: 860 (29.8% positivo) | teste: 1014 (24.2% positivo)
ROC-AUC: 0.629 | PR-AUC: 0.390
              precision    recall  f1-score   support

           0       0.79      0.91      0.85       769
           1       0.47      0.26      0.33       245

    accuracy                           0.75      1014
   macro avg       0.63      0.58      0.59      1014
weighted avg       0.71      0.75      0.72      1014



## Explicabilidade (SHAP)

Agrupamos as features em 4 eixos pedagógicos e, para cada aluno, apontamos o
eixo que mais empurrou o risco para cima — o "driver dominante" que vai para
o dashboard de triagem.

In [6]:
from tc5.explain import AXES, driver_dominante

assert sorted(sum(AXES.values(), [])) == sorted(FEATURE_COLS), "toda feature precisa estar em um eixo"

In [7]:
drivers_def = driver_dominante(model_defasagem, Xte_def)
drivers_ev = driver_dominante(model_evasao, Xte_ev)

print("driver dominante — risco de defasagem (teste 2023):")
print(drivers_def["driver_dominante"].value_counts(normalize=True).round(2))
print("\ndriver dominante — risco de evasão (teste 2023):")
print(drivers_ev["driver_dominante"].value_counts(normalize=True).round(2))

driver dominante — risco de defasagem (teste 2023):
driver_dominante
academico          0.92
contexto           0.06
psicopedagogico    0.02
Name: proportion, dtype: float64

driver dominante — risco de evasão (teste 2023):
driver_dominante
academico          0.61
engajamento        0.31
contexto           0.07
psicopedagogico    0.01
Name: proportion, dtype: float64


In [8]:
import joblib

joblib.dump(model_defasagem, "../data/processed/model_defasagem.joblib")
joblib.dump(model_evasao, "../data/processed/model_evasao.joblib")

risco = te_ev[["ra", "ano"]].copy()
risco["risco_evasao"] = model_evasao.predict_proba(Xte_ev)[:, 1]
risco["driver_evasao"] = drivers_ev["driver_dominante"].values
risco = risco.merge(
    pd.DataFrame({
        "ra": te_def["ra"].values,
        "risco_defasagem": model_defasagem.predict_proba(Xte_def)[:, 1],
        "driver_defasagem": drivers_def["driver_dominante"].values,
    }),
    on="ra", how="outer",
)
risco.to_csv("../data/processed/risco_alunos.csv", index=False)
risco.sort_values("risco_evasao", ascending=False).head(10)

,ra,ano,risco_evasao,driver_evasao,risco_defasagem,driver_defasagem
188,RA-1175,2023,0.972166,engajamento,NaN,NaN
540,RA-468,2023,0.960764,engajamento,NaN,NaN
142,RA-1132,2023,0.918021,engajamento,NaN,NaN
398,RA-285,2023,0.910857,engajamento,NaN,NaN
699,RA-665,2023,0.909077,engajamento,0.072508,academico
238,RA-1222,2023,0.907705,academico,NaN,NaN
117,RA-1109,2023,0.898750,engajamento,NaN,NaN
343,RA-192,2023,0.894105,engajamento,0.019519,academico
231,RA-1215,2023,0.887377,engajamento,NaN,NaN
983,RA-971,2023,0.884256,engajamento,0.779149,academico


## Conclusões

- **`piora_defasagem`**: ROC-AUC 0.83, PR-AUC 0.52. Bom — trajetória
  acadêmica passada (`defasagem`, `ida`, notas) prediz bem a trajetória
  futura, como esperado. Driver dominante é quase sempre `academico` (92%).
- **`evasao`**: ROC-AUC 0.63, PR-AUC 0.39. Sensivelmente mais fraco — a
  matriz PEDE mede desempenho e engajamento acadêmico, não capta os
  motivos mais comuns de evasão em ONGs (mudança de endereço, questão
  financeira da família, etc.). Esperado, não é falha de modelagem: o sinal
  disponível é parcial.
- Eixo `psicossocial` nunca aparece como driver dominante em nenhum dos dois
  modelos — indício de que `ips` carrega pouco sinal incremental aqui,
  vale investigar no storytelling (Q5 do enunciado, ligada a IPS).

**Implicação para o dashboard**: tratar `risco_evasao` como sinalizador para
priorização humana, não como veredito — reforça a proposta do dashboard de
triagem (lista ranqueada + driver) em vez de um score isolado.